# Exercise 1: Russian documents and scripts

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Meaning and Retrieval · version 0.1.0*

> Extend the hands-on to SIB-200: for Kazakh, Kyrgyz and Tatar queries, retrieve the Russian translation of each sentence and compare with Turkic neighbours. Then test native/native, CTS/CTS and native→CTS conditions for four pairs.

**Setup.** `pip install torch transformers sentencepiece scipy bm25s requests "turkicnlp[hfst]==0.5.0"` (the compiled `hfst` package needs Python 3.11 or 3.12). Encoders are loaded at the Hugging Face revisions pinned in `MODELS`, from a local Hugging Face cache (`HF_HUB_DIR`), a folder of copies (`MODELS_DIR`) or the Hub; NLLB-200 is read from TurkicNLP's model folder (`~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M`) as the `embeddings` processor does. Data come from commit-pinned GitHub repositories: SIB-200 (FLORES-200 sentences, the same 1,004 sentences in every language), Kardeş-NLU's STS sets and KazQAD's retrieval collection (825,309 Kazakh Wikipedia passages, about 230 MB compressed). Embeddings, lemma maps and retrieval runs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`). Everything runs on the CPU.

SIB-200 contains 1,004 FLORES-200 sentences, identical in content across its languages and freely available on GitHub. Each query sentence is compared with all 1,004 sentences of the target language; top-1 accuracy is the share whose nearest neighbour is the translation. The encoders are the two paraphrase models of the chapter's measured table and the NLLB-200 encoder that TurkicNLP's `embeddings` processor uses (mean pooling over the last encoder layer).

In [ ]:
import collections, csv, gzip, hashlib, io, json, math, os, random, re, statistics, time
from pathlib import Path
import numpy as np, requests

SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
KARDES = "https://raw.githubusercontent.com/lksenel/Kardes-NLU/48eea77fda904e731d228de833f4c57d130fad9c/Data/"
KAZQAD = "https://raw.githubusercontent.com/IS2AI/KazQAD/ab8684a667109f2a7345f4f89ab3b123e8d78e1f/data/information-retrieval/"
MODELS = {   # sentence encoder -> pinned revision (mean pooling, 128-token ceiling, as configured)
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
    "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a",
    "cis-lmu/glot500-base": "d4d7c1ec01828fdf7452a4ccf7b55177aced175e",                 # tokenizer only
    "meta-llama/Meta-Llama-3-8B": "8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920",          # tokenizer only
    "allenai/Olmo-3-1025-7B": "0a15c289b17ad4f27aabc863276ec525d5312c01",              # tokenizer only
    "Qwen/Qwen3.6-27B": "6a9e13bd6fc8f0983b9b99948120bc37f49c13e9",                    # tokenizer only
}
NAMES = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "MiniLM",
         "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "XLM-R para", "nllb": "NLLB enc"}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
HF_HUB_DIR = Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser()
MODELS_DIR = Path(os.environ.get("MODELS_DIR", CACHE / "models"))
NLLB_DIR = Path("~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M").expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=600); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    FETCHED[url] = h.hexdigest()
    return path

def model_source(name):
    rev = MODELS[name]
    for p in (HF_HUB_DIR / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              MODELS_DIR / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev

def tokenizer(name):
    from transformers import AutoTokenizer
    path, rev = model_source(name)
    return AutoTokenizer.from_pretrained(path, revision=rev)

def sib(lang):
    """All 1,004 SIB-200 sentences of a FLORES language (train, dev, test), sorted by index_id."""
    rows = []
    for split in ("train", "dev", "test"):
        text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
        rows += [(int(r["index_id"]), r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]
    return [t for _, t in sorted(rows)]

TRANSLIT = {"kaz_Cyrl": ("kaz", "Cyrl"), "kir_Cyrl": ("kir", "Cyrl"), "tat_Cyrl": ("tat", "Cyrl"),
            "bak_Cyrl": ("bak", "Cyrl"), "uzn_Latn": ("uzb", "Latn"), "azj_Latn": ("aze", "Latn"), "tur_Latn": ("tur", "Latn")}
_T = {}
def to_cts(lang, text):
    """Common Turkic Alphabet version of a sentence with TurkicNLP's transliterator."""
    if lang not in _T:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        code, scr = TRANSLIT[lang]
        _T[lang] = Transliterator(code, Script.CYRILLIC if scr == "Cyrl" else Script.LATIN, Script.COMMON_TURKIC)
    return _T[lang].transliterate(text)

_ENC = {}
NLLB_CODE = {"kaz_Cyrl": "kaz_Cyrl", "kir_Cyrl": "kir_Cyrl", "tat_Cyrl": "tat_Cyrl", "bak_Cyrl": "bak_Cyrl",
             "uzn_Latn": "uzn_Latn", "azj_Latn": "azj_Latn", "tur_Latn": "tur_Latn", "rus_Cyrl": "rus_Cyrl",
             "eng_Latn": "eng_Latn", "uig_Arab": "uig_Arab"}
def embed(model, texts, lang=None, batch=32):
    """L2-normalised mean-pooled sentence vectors. Paraphrase encoders truncate at their configured
    128 tokens; the NLLB encoder (as in TurkicNLP's embeddings processor) gets the language token of
    `lang` and is not truncated below 512. Cached on disk by model, language and text hash."""
    key = hashlib.sha1(json.dumps([model, lang, texts], ensure_ascii=False).encode()).hexdigest()[:20]
    path = CACHE / "ch13-vectors" / f"{key}.npy"
    if path.exists():
        return np.load(path)
    import torch
    torch.set_num_threads(THREADS)
    if model not in _ENC:
        from transformers import AutoModel, AutoTokenizer
        if model == "nllb":
            from transformers import AutoModelForSeq2SeqLM
            tok = AutoTokenizer.from_pretrained(NLLB_DIR)
            enc = AutoModelForSeq2SeqLM.from_pretrained(NLLB_DIR).get_encoder().eval()
            _ENC[model] = (tok, enc, 512)
        else:
            src, rev = model_source(model)
            _ENC[model] = (AutoTokenizer.from_pretrained(src, revision=rev), AutoModel.from_pretrained(src, revision=rev).eval(), 128)
    tok, enc, limit = _ENC[model]
    if model == "nllb":
        tok.src_lang = NLLB_CODE[lang]
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], padding=True, truncation=True, max_length=limit, return_tensors="pt")
            h = enc(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            v = (h * m).sum(1) / m.sum(1).clamp(min=1)
            out.append(torch.nn.functional.normalize(v, dim=-1).numpy())
    vec = np.concatenate(out).astype(np.float32)
    path.parent.mkdir(parents=True, exist_ok=True); np.save(path, vec)
    return vec

def top1(q, c, targets):
    """Share of queries whose nearest candidate is the aligned one (exact duplicate target texts count)."""
    best = (q @ c.T).argmax(1)
    return float(np.mean([targets[b] == targets[i] for i, b in enumerate(best)]))

print("encoders:", ", ".join(f"{NAMES.get(n, n)} ({'local' if model_source(n)[1] is None else 'Hub'})" for n in list(MODELS)[:2]),
      f"+ NLLB encoder ({'local' if (NLLB_DIR / 'config.json').exists() else 'missing'})")

In [ ]:
LANGS = ["kaz_Cyrl", "kir_Cyrl", "tat_Cyrl", "bak_Cyrl", "uzn_Latn", "tur_Latn", "rus_Cyrl", "eng_Latn"]
TEXT = {l: sib(l) for l in LANGS}
assert len({len(v) for v in TEXT.values()}) == 1
ENCODERS = ["sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "sentence-transformers/paraphrase-xlm-r-multilingual-v1", "nllb"]
V = {(m, l): embed(m, TEXT[l], l) for m in ENCODERS for l in LANGS}
print(f"top-1 retrieval among {len(TEXT['kaz_Cyrl']):,} SIB-200 (FLORES-200) sentences; rows: query language, columns: candidates")
TARGETS = ["rus_Cyrl", "eng_Latn", "kir_Cyrl", "kaz_Cyrl", "tat_Cyrl", "uzn_Latn", "tur_Latn"]
for m in ENCODERS:
    print(f"\n{NAMES[m]:<10} " + "".join(f"{t[:3]:>7}" for t in TARGETS))
    for q in ("kaz_Cyrl", "kir_Cyrl", "tat_Cyrl"):
        print(f"  {q:<9}" + "".join("      -" if t == q else f"{top1(V[m, q], V[m, t], TEXT[t]):7.1%}" for t in TARGETS))

# Script conditions: native/native, CTS/CTS (both sides converted) and native query -> CTS candidates.
PAIRS = [("kaz_Cyrl", "kir_Cyrl"), ("kaz_Cyrl", "tur_Latn"), ("kaz_Cyrl", "uzn_Latn"), ("tat_Cyrl", "bak_Cyrl")]
CTS = {l: [to_cts(l, s) for s in TEXT[l]] for l in {x for p in PAIRS for x in p}}
print("\nCTS conversion leaves unchanged:", {l: f"{sum(a == b for a, b in zip(TEXT[l], CTS[l]))}/{len(CTS[l])}" for l in CTS})
print("example:", CTS["kaz_Cyrl"][0][:80])
print(f"\n{'pair':<20}{'encoder':<11}{'native':>8}{'CTS/CTS':>9}{'nat->CTS':>10}")
for a, b in PAIRS:
    for m in ENCODERS:
        na = top1(V[m, a], V[m, b], TEXT[b])
        ca, cb = embed(m, CTS[a], a), embed(m, CTS[b], b)
        print(f"{a[:3]}->{b[:3]:<14}{NAMES[m]:<11}{na:8.1%}{top1(ca, cb, CTS[b]):9.1%}{top1(V[m, a], cb, CTS[b]):10.1%}")

## Worked solution

Output of `experiments/ch13_exercises/run_ch13_exercises.py`:

```
top-1 retrieval among 1,004 SIB-200 (FLORES-200) sentences; rows: query language, columns: candidates

MiniLM         rus    eng    kir    kaz    tat    uzn    tur
  kaz_Cyrl   43.9%  47.1%  41.7%      -  28.4%  25.8%  45.9%
  kir_Cyrl   50.6%  53.1%      -  47.4%  36.2%  32.2%  52.4%
  tat_Cyrl   43.8%  43.6%  39.4%  33.6%      -  31.5%  43.0%

XLM-R para     rus    eng    kir    kaz    tat    uzn    tur
  kaz_Cyrl   97.8%  97.5%  93.7%      -  58.4%  65.8%  97.1%
  kir_Cyrl   94.6%  94.4%      -  91.7%  58.4%  65.2%  95.1%
  tat_Cyrl   56.7%  55.2%  48.6%  49.7%      -  42.9%  56.0%

NLLB enc       rus    eng    kir    kaz    tat    uzn    tur
  kaz_Cyrl   99.6%  99.9%  99.6%      -  99.6%  99.2%  99.6%
  kir_Cyrl   99.4% 100.0%      -  98.9%  99.6%  99.6%  99.5%
  tat_Cyrl   99.9%  99.9%  99.6%  99.2%      -  99.7% 100.0%

CTS conversion leaves unchanged: {'tur_Latn': '1004/1004', 'kaz_Cyrl': '0/1004', 'bak_Cyrl': '0/1004', 'uzn_Latn': '2/1004', 'tat_Cyrl': '0/1004', 'kir_Cyrl': '0/1004'}
example: Düysenbi küni Stenford universitetiniñ meditsina fakultetiniñ ğalımdarı casuşala

pair                encoder      native  CTS/CTS  nat->CTS
kaz->kir           MiniLM        41.7%    45.4%     24.7%
kaz->kir           XLM-R para    93.7%    46.1%     52.8%
kaz->kir           NLLB enc      99.6%    46.0%     30.8%
kaz->tur           MiniLM        45.9%    36.3%     45.9%
kaz->tur           XLM-R para    97.1%    42.2%     97.1%
kaz->tur           NLLB enc      99.6%    25.1%     99.6%
kaz->uzn           MiniLM        25.8%    35.9%     23.0%
kaz->uzn           XLM-R para    65.8%    37.9%     62.8%
kaz->uzn           NLLB enc      99.2%    22.1%     98.2%
tat->bak           MiniLM        54.1%    54.1%     33.4%
tat->bak           XLM-R para    56.2%    55.8%     41.7%
tat->bak           NLLB enc      99.6%    49.5%     15.3%
```

**Native-script matching.** XLM-R retrieves the Russian translation for 97.8% of Kazakh queries, compared with 93.7% for Kyrgyz candidates. Tatar scores are lower in this configuration. These are descriptive comparisons, not evidence that genealogy or pretraining coverage caused the gaps. NLLB scores 98.9–100% across the native-script directions shown; high performance and the dataset’s origin do not prove memorisation. Independent evaluation remains needed.

**CTS conditions.** On Kazakh→Kyrgyz, CTS/CTS lowers XLM-R from 93.7% to 46.1% and NLLB from 99.6% to 46.0%, while MiniLM rises from 41.7% to 45.4%. The results do not isolate the roles of tokenisation, mapping losses or pretraining. Turkish is unchanged by its converter; only 2 of 1,004 Uzbek sentences are unchanged. Native→CTS is therefore not a uniform transformation across targets.

**Protocol limits.** Each cell uses 1,004 queries and candidates. The loader sorts each language by `index_id` but does not assert matching unique ID sets. Exact duplicate targets count as correct; `argmax` resolves ties by the first candidate. The paraphrase encoders truncate at 128 tokens and NLLB at 512. NLLB retains the original language token for CTS input. These differ from the stricter hands-on loader and tie policy. Audit alignment, truncation and model inputs before extending the comparison.

## Your turn — extension

1. Add Uyghur (`uig_Arab`) and Azerbaijani (`azj_Latn`) queries and compare Russian and Turkish targets.
2. Compare the original NLLB language token with `eng_Latn` on development data. The latter requests the English token; it does not remove language conditioning. Treat this as a separate diagnostic, not an established CTS configuration.